# 📦 Notebook 02 — Embedding, ChromaDB & TF-IDF Index Build

**Run on Google Colab (T4 GPU)**

This notebook takes the cleaned parquet from Notebook 01 and:

| Step | What | Output |
|------|------|--------|
| 0 | Install dependencies | — |
| 1 | Mount Drive & load parquet | DataFrame |
| 2 | Verify GPU | — |
| 3 | Load Jina v3 embedding model | Model on GPU |
| 4 | Batch-embed `embedding_text` | `embeddings.npy` |
| 5 | Create ChromaDB collection | `hadith_chroma_db/` |
| 6 | Bulk insert into ChromaDB | — |
| 7 | Build TF-IDF sparse index | `tfidf_index.pkl` |
| 8 | Verification & test queries | — |
| 9 | Zip & download artifacts | `hadith_rag_artifacts.zip` |

**Output artifacts** (download to local `data/` and `chroma_db/`):
- `hadith_chroma_db/` — ChromaDB vector store
- `tfidf_index.pkl` — TF-IDF sparse index
- `dataset_stats.json` — pre-computed stats (already from notebook 01)

In [ ]:
# Step 0: Install dependencies
!pip install -q \
    torch \
    transformers \
    chromadb \
    pandas \
    pyarrow \
    scikit-learn \
    joblib \
    tqdm

In [ ]:
# Step 1: Mount Drive & load cleaned parquet
from google.colab import drive
drive.mount('/content/drive')

import os, json
import pandas as pd
import numpy as np

DATA_DIR = '/content/drive/MyDrive/colabnotebooks/yaqeen_hadith_rag/data'
PARQUET_PATH = f'{DATA_DIR}/cleaned_hadiths.parquet'

if not os.path.exists(PARQUET_PATH):
    raise FileNotFoundError(
        f"\n❌ {PARQUET_PATH} not found.\n"
        "Run Notebook 01 first to generate cleaned_hadiths.parquet."
    )

df = pd.read_parquet(PARQUET_PATH)
print(f"✅ Loaded {len(df):,} hadiths from parquet")
print(f"   Columns: {list(df.columns)}")
print(f"\n--- Sample embedding_text ---")
print(df['embedding_text'].iloc[0][:250])

In [ ]:
# Step 2: Verify GPU
import torch

if not torch.cuda.is_available():
    print("⚠️  WARNING: No GPU detected. Embedding will be very slow!")
    print("   Go to Runtime → Change runtime type → T4 GPU")
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"   CUDA: {torch.version.cuda}")
    print(f"   PyTorch: {torch.__version__}")

In [ ]:
# Step 3: Load Jina v3 embedding model
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = "jinaai/jina-embeddings-v3"
print(f"Loading {MODEL_NAME} …")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = model.to('cuda').half()  # fp16 for speed
model.eval()

# Quick sanity check
test_out = model.encode(["بسم الله الرحمن الرحيم"], task="retrieval.passage")
print(f"✅ Model loaded — embedding dim: {test_out.shape[-1]}")
print(f"   Device: {next(model.parameters()).device}")

In [ ]:
# Step 4: Batch-embed the metadata-enriched text
from tqdm.auto import tqdm
import gc

texts = df['embedding_text'].tolist()
BATCH_SIZE = 64
EMBED_DIM  = 1024
all_embeddings = np.zeros((len(texts), EMBED_DIM), dtype=np.float32)

print(f"Embedding {len(texts):,} texts in batches of {BATCH_SIZE} …")

for start in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[start : start + BATCH_SIZE]
    with torch.no_grad():
        emb = model.encode(batch, task="retrieval.passage")
    if isinstance(emb, torch.Tensor):
        emb = emb.cpu().numpy()
    all_embeddings[start : start + len(batch)] = emb.astype(np.float32)

    # Periodic GPU cache clear
    if (start // BATCH_SIZE) % 50 == 0:
        torch.cuda.empty_cache()

# Save to Drive (safety checkpoint)
emb_path = f'{DATA_DIR}/embeddings.npy'
np.save(emb_path, all_embeddings)
emb_size = os.path.getsize(emb_path) / (1024**2)

print(f"\n✅ Embeddings complete!")
print(f"   Shape: {all_embeddings.shape}")
print(f"   Saved: {emb_path} ({emb_size:.1f} MB)")
print(f"   Norm range: [{np.linalg.norm(all_embeddings, axis=1).min():.4f}, {np.linalg.norm(all_embeddings, axis=1).max():.4f}]")

In [ ]:
# Step 5: Create ChromaDB collection
import chromadb

CHROMA_DIR = f'{DATA_DIR}/hadith_chroma_db'

# Delete old DB if it exists
if os.path.exists(CHROMA_DIR):
    import shutil
    shutil.rmtree(CHROMA_DIR)
    print(f"🗑️  Removed old ChromaDB at {CHROMA_DIR}")

client = chromadb.PersistentClient(path=CHROMA_DIR)

collection = client.create_collection(
    name="hadiths",
    metadata={
        "hnsw:space": "cosine",
        "hnsw:construction_ef": 200,
        "hnsw:M": 32,
        "hnsw:search_ef": 150,
    },
)

print(f"✅ Created ChromaDB collection 'hadiths' at {CHROMA_DIR}")
print(f"   HNSW config: cosine, ef_construction=200, M=32, ef_search=150")

In [ ]:
# Step 6: Bulk insert into ChromaDB (batches of 5000)
INSERT_BATCH = 5000
total = len(df)

print(f"Inserting {total:,} documents into ChromaDB …")

for start in tqdm(range(0, total, INSERT_BATCH)):
    end = min(start + INSERT_BATCH, total)
    batch_df = df.iloc[start:end]

    ids = batch_df['doc_id'].tolist()

    # Documents = embedding_text (same text we embedded)
    documents = batch_df['embedding_text'].tolist()

    # Embeddings
    embeddings = all_embeddings[start:end].tolist()

    # Metadata — all fields needed at retrieval time
    metadatas = []
    for _, row in batch_df.iterrows():
        meta = {
            "text_ar":             str(row.get('text_ar', '')),
            "text_ar_raw":         str(row.get('text_ar_raw', '')),
            "grade":               str(row.get('grade', 'unknown')),
            "grade_ar":            str(row.get('grade_ar', 'غير معروف')),
            "ruling":              str(row.get('ruling', '')),
            "rawi":                str(row.get('rawi', '')),
            "mohadeth":            str(row.get('mohadeth', '')),
            "book":                str(row.get('book', '')),
            "numberOrPage":        str(row.get('numberOrPage', '')),
            "hadith_tag":          str(row.get('hadith_tag', '')),
            "hasExplanation":      bool(row.get('hasExplanation', False)),
            "canonical_group_id":  str(row.get('canonical_group_id', '')),
        }
        metadatas.append(meta)

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas,
    )

print(f"\n✅ ChromaDB insert complete!")
print(f"   Collection count: {collection.count():,}")

In [ ]:
# Step 7: Build TF-IDF sparse index
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

print("Building TF-IDF index on text_ar (char n-grams 3-5) …")

tfidf_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=150_000,
    sublinear_tf=True,
    min_df=2,
    max_df=0.95,
    dtype=np.float32,
)

tfidf_texts = df['text_ar'].tolist()
tfidf_matrix = tfidf_vectorizer.fit_transform(tfidf_texts)

# Save the vectorizer + matrix + doc_ids together
tfidf_index = {
    'vectorizer': tfidf_vectorizer,
    'matrix': tfidf_matrix,
    'doc_ids': df['doc_id'].tolist(),
}

tfidf_path = f'{DATA_DIR}/tfidf_index.pkl'
joblib.dump(tfidf_index, tfidf_path, compress=3)

tfidf_size = os.path.getsize(tfidf_path) / (1024**2)
print(f"\n✅ TF-IDF index built!")
print(f"   Vocab size:    {len(tfidf_vectorizer.vocabulary_):,}")
print(f"   Matrix shape:  {tfidf_matrix.shape}")
print(f"   Saved: {tfidf_path} ({tfidf_size:.1f} MB)")

In [ ]:
# Step 8: Verification & test queries

print("=" * 60)
print("VERIFICATION")
print("=" * 60)

# 8a. ChromaDB count
print(f"\n📦 ChromaDB collection count: {collection.count():,}")

# 8b. Test dense query
test_query = "أحاديث عن الصبر"
with torch.no_grad():
    q_emb = model.encode([test_query], task="retrieval.query")
if isinstance(q_emb, torch.Tensor):
    q_emb = q_emb.cpu().numpy()

results = collection.query(
    query_embeddings=q_emb.tolist(),
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print(f"\n🔍 Dense test: '{test_query}'")
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], results['metadatas'][0], results['distances'][0]
)):
    print(f"\n  [{i+1}] distance={dist:.4f} | grade={meta['grade']} | rawi={meta['rawi'][:30]}")
    print(f"      {doc[:150]}…")

# 8c. Test metadata filter
results_filter = collection.query(
    query_embeddings=q_emb.tolist(),
    n_results=3,
    where={"grade": "sahih"},
    include=["metadatas", "distances"],
)
print(f"\n🔍 Filtered (sahih only): {len(results_filter['ids'][0])} results")

# 8d. Test TF-IDF
from sklearn.metrics.pairwise import cosine_similarity
q_tfidf = tfidf_vectorizer.transform([test_query])
sims = cosine_similarity(q_tfidf, tfidf_matrix).flatten()
top_idx = sims.argsort()[-3:][::-1]
print(f"\n🔍 TF-IDF test: '{test_query}'")
for idx in top_idx:
    print(f"  score={sims[idx]:.4f} | {df['text_ar'].iloc[idx][:100]}…")

print(f"\n✅ All verification passed!")

In [ ]:
# Step 9: Zip artifacts & download

import shutil

# Free GPU memory
del model, tokenizer
torch.cuda.empty_cache()
gc.collect()

ARTIFACT_DIR = '/content/hadith_rag_artifacts'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# Copy ChromaDB
dest_chroma = f'{ARTIFACT_DIR}/hadith_chroma_db'
if os.path.exists(dest_chroma):
    shutil.rmtree(dest_chroma)
shutil.copytree(CHROMA_DIR, dest_chroma)
print(f"✅ Copied ChromaDB → {dest_chroma}")

# Copy TF-IDF index
shutil.copy2(tfidf_path, f'{ARTIFACT_DIR}/tfidf_index.pkl')
print(f"✅ Copied TF-IDF index")

# Copy dataset_stats.json
stats_src = f'{DATA_DIR}/dataset_stats.json'
if os.path.exists(stats_src):
    shutil.copy2(stats_src, f'{ARTIFACT_DIR}/dataset_stats.json')
    print(f"✅ Copied dataset_stats.json")

# Create zip
zip_path = '/content/hadith_rag_artifacts'
shutil.make_archive(zip_path, 'zip', ARTIFACT_DIR)
zip_size = os.path.getsize(f'{zip_path}.zip') / (1024**2)
print(f"\n📦 Created: {zip_path}.zip ({zip_size:.1f} MB)")

# Also save zip to Drive
drive_zip = f'{DATA_DIR}/hadith_rag_artifacts.zip'
shutil.copy2(f'{zip_path}.zip', drive_zip)
print(f"📦 Backed up to Drive: {drive_zip}")

# Download
from google.colab import files
print(f"\n⬇️  Starting download …")
files.download(f'{zip_path}.zip')

print(f"""
🎉 Done! After downloading, extract the zip into your local project:

  hadith_rag/
  ├── chroma_db/hadith_chroma_db/   ← from zip: hadith_chroma_db/
  ├── data/tfidf_index.pkl          ← from zip: tfidf_index.pkl
  └── data/dataset_stats.json       ← from zip: dataset_stats.json
""")